In [2]:
import pandas as pd
import numpy as np
import seaborn as sns
import matplotlib.pyplot as plt
import statsmodels.api as sm
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import classification_report, confusion_matrix

%matplotlib inline
pd.options.display.float_format = '{:.2f}'.format 

In [3]:
dfraw = pd.read_csv('Raw_Data[1].csv')

In [4]:
#inspect data
print(dfraw); obs=10

                                Name Platform  Year_of_Release         Genre  \
0                         Wii Sports      Wii          2006.00        Sports   
1                  Super Mario Bros.      NES          1985.00      Platform   
2                     Mario Kart Wii      Wii          2008.00        Racing   
3                  Wii Sports Resort      Wii          2009.00        Sports   
4           Pokemon Red/Pokemon Blue       GB          1996.00  Role-Playing   
...                              ...      ...              ...           ...   
16714  Samurai Warriors: Sanada Maru      PS3          2016.00        Action   
16715               LMA Manager 2007     X360          2006.00        Sports   
16716        Haitaka no Psychedelica      PSV          2016.00     Adventure   
16717               Spirits & Spells      GBA          2003.00      Platform   
16718            Winning Post 8 2016      PSV          2016.00    Simulation   

          Publisher  NA_Sales  EU_Sales

In [5]:
print(dfraw.shape)
print(dfraw.dtypes)

(16719, 16)
Name                object
Platform            object
Year_of_Release    float64
Genre               object
Publisher           object
NA_Sales           float64
EU_Sales           float64
JP_Sales           float64
Other_Sales        float64
Global_Sales       float64
Critic_Score       float64
Critic_Count       float64
User_Score          object
User_Count         float64
Developer           object
Rating              object
dtype: object


In [6]:
dfraw.isnull().sum()

Name                  2
Platform              0
Year_of_Release     269
Genre                 2
Publisher            54
NA_Sales              0
EU_Sales              0
JP_Sales              0
Other_Sales           0
Global_Sales          0
Critic_Score       8582
Critic_Count       8582
User_Score         6704
User_Count         9129
Developer          6623
Rating             6769
dtype: int64

In [7]:
dfraw.describe(include='all')

,Name,Platform,Year_of_Release,Genre,Publisher,NA_Sales,EU_Sales,JP_Sales,Other_Sales,Global_Sales,Critic_Score,Critic_Count,User_Score,User_Count,Developer,Rating
count,16717,16719,16450.00,16717,16665,16719.00,16719.00,16719.00,16719.00,16719.00,8137.00,8137.00,10015,7590.00,10096,9950
unique,11562,31,NaN,12,581,NaN,NaN,NaN,NaN,NaN,NaN,NaN,96,NaN,1696,8
top,Need for Speed: Most Wanted,PS2,NaN,Action,Electronic Arts,NaN,NaN,NaN,NaN,NaN,NaN,NaN,tbd,NaN,Ubisoft,E
freq,12,2161,NaN,3370,1356,NaN,NaN,NaN,NaN,NaN,NaN,NaN,2425,NaN,204,3991
mean,NaN,NaN,2006.49,NaN,NaN,0.26,0.15,0.08,0.05,0.53,68.97,26.36,NaN,162.23,NaN,NaN
std,NaN,NaN,5.88,NaN,NaN,0.81,0.50,0.31,0.19,1.55,13.94,18.98,NaN,561.28,NaN,NaN
min,NaN,NaN,1980.00,NaN,NaN,0.00,0.00,0.00,0.00,0.01,13.00,3.00,NaN,4.00,NaN,NaN
25%,NaN,NaN,2003.00,NaN,NaN,0.00,0.00,0.00,0.00,0.06,60.00,12.00,NaN,10.00,NaN,NaN
50%,NaN,NaN,2007.00,NaN,NaN,0.08,0.02,0.00,0.01,0.17,71.00,21.00,NaN,24.00,NaN,NaN
75%,NaN,NaN,2010.00,NaN,NaN,0.24,0.11,0.04,0.03,0.47,79.00,36.00,NaN,81.00,NaN,NaN


In [8]:
#remove missing values
df_clean = dfraw.dropna()
print(f"Original shape: {dfraw.shape}")
print(f"Cleaned shape: {df_clean.shape}")


Original shape: (16719, 16)
Cleaned shape: (6825, 16)


In [9]:
df_clean.head()
df_clean['Global_Sales'].describe()

count   6825.00
mean       0.78
std        1.96
min        0.01
25%        0.11
50%        0.29
75%        0.75
max       82.53
Name: Global_Sales, dtype: float64

In [10]:
#categorize top seller
q75 = df_clean['Global_Sales'].quantile(0.75)
print(f"75th percentile of Global_Sales: {q75:.2f} million units")

df_clean['Top_Seller_75'] = (df_clean['Global_Sales'] > q75).astype(int)
print(df_clean['Top_Seller_75'].value_counts(normalize=True))


75th percentile of Global_Sales: 0.75 million units
Top_Seller_75
0   0.75
1   0.25
Name: proportion, dtype: float64


/tmp/ipykernel_254/3565316007.py:5: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  df_clean['Top_Seller_75'] = (df_clean['Global_Sales'] > q75).astype(int)


In [11]:
from sklearn.preprocessing import LabelEncoder

In [12]:
df_model = df_clean.copy()

# Convert User_Score to numeric
df_model['User_Score'] = pd.to_numeric(df_model['User_Score'], errors='coerce')

# Label encode categorical columns
cat_cols = ['Platform', 'Genre', 'Publisher', 'Developer', 'Rating']
for col in cat_cols:
    df_model[col] = LabelEncoder().fit_transform(df_model[col].astype(str))


In [13]:
if 'Name' in df_model.columns:
    df_model = df_model.drop(columns=['Name'])
    print(df_model.dtypes)

Platform             int64
Year_of_Release    float64
Genre                int64
Publisher            int64
NA_Sales           float64
EU_Sales           float64
JP_Sales           float64
Other_Sales        float64
Global_Sales       float64
Critic_Score       float64
Critic_Count       float64
User_Score         float64
User_Count         float64
Developer            int64
Rating               int64
Top_Seller_75        int64
dtype: object


In [14]:
sales_cols = ['NA_Sales', 'EU_Sales', 'JP_Sales', 'Other_Sales','Global_Sales']
df_model = df_model.drop(columns=sales_cols)

In [15]:
from sklearn.model_selection import train_test_split

# features and target
X = df_model.drop(columns=['Top_Seller_75'])
y = df_model['Top_Seller_75']

# split data
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)


In [16]:
from sklearn.model_selection import cross_val_score, cross_validate
from sklearn.linear_model import LogisticRegression
from sklearn.pipeline import make_pipeline
from sklearn.preprocessing import StandardScaler

# Create a pipeline: StandardScaler + Logistic Regression
logreg_pipeline = make_pipeline(
    StandardScaler(),
    LogisticRegression(max_iter=1000, class_weight='balanced')
)

# Fit and predict
logreg_pipeline.fit(X_train, y_train)
y_pred_log = logreg_pipeline.predict(X_test)

# Evaluate
from sklearn.metrics import classification_report
print(classification_report(y_test, y_pred_log))


              precision    recall  f1-score   support

           0       0.88      0.73      0.80      1004
           1       0.50      0.73      0.59       361

    accuracy                           0.73      1365
   macro avg       0.69      0.73      0.69      1365
weighted avg       0.78      0.73      0.74      1365



In [17]:
# Define pipeline
pipeline = make_pipeline(
    StandardScaler(),
    LogisticRegression(max_iter=1000, class_weight='balanced')
)

# Define scoring metrics
scoring = ['accuracy', 'precision', 'recall', 'f1']

# Run 5-fold cross-validation
cv_results = cross_validate(pipeline, X, y, cv=5, scoring=scoring)

# Display average results
print("Cross-Validation Results (Logistic Regression):")
for metric in scoring:
    mean_score = np.mean(cv_results[f'test_{metric}'])
    print(f"{metric.capitalize()}: {mean_score:.3f}")

Cross-Validation Results (Logistic Regression):
Accuracy: 0.713
Precision: 0.474
Recall: 0.703
F1: 0.552


In [ ]:
rf = RandomForestClassifier(random_state=42, class_weight='balanced')
rf.fit(X_train, y_train)
y_pred_rf = rf.predict(X_test)

print("Random Forest Report:")
print(classification_report(y_test, y_pred_rf))

In [ ]:
importances = rf.feature_importances_
features = X.columns
importance_df = pd.DataFrame({'Feature': features, 'Importance': importances})
importance_df = importance_df.sort_values('Importance', ascending=False)

# Plot
plt.figure(figsize=(10, 6))
sns.barplot(x='Importance', y='Feature', data=importance_df)
plt.title("Feature Importance (Random Forest)")
plt.tight_layout()
plt.show()

In [ ]:
from sklearn.metrics import roc_curve, auc

# Get predicted probabilities
y_probs = logreg_pipeline.predict_proba(X_test)[:, 1]
fpr, tpr, thresholds = roc_curve(y_test, y_probs)
roc_auc = auc(fpr, tpr)

# Plot
plt.figure(figsize=(8, 5))
plt.plot(fpr, tpr, label=f'AUC = {roc_auc:.2f}')
plt.plot([0, 1], [0, 1], linestyle='--', color='gray')
plt.xlabel('False Positive Rate')
plt.ylabel('True Positive Rate')
plt.title('ROC Curve (Logistic Regression)')
plt.legend()
plt.grid(True)
plt.tight_layout()
plt.show()


In [ ]:
df_model_temp = df_model.copy()
df_model_temp['Top_Seller_75'] = y

plt.figure(figsize=(12, 8))
sns.heatmap(df_model_temp.corr(), annot=True, cmap='coolwarm')
plt.title('Correlation Matrix')
plt.tight_layout()
plt.show()

In [ ]:

df_clean.groupby('Publisher')['Top_Seller_75'].mean().sort_values(ascending=False).head(10)


In [ ]:


df_clean.groupby('Genre')['Top_Seller_75'].mean().sort_values(ascending=False)


In [ ]:
multiple_columns_counts = df_clean.groupby(['Genre', 'Publisher']).size().reset_index(name='count')

print(multiple_columns_counts)

In [ ]:
genre_counts = df_clean['Genre'].value_counts()
publisher_counts=df_clean['Publisher'].value_counts()

print(genre_counts)
print(publisher_counts)

In [ ]:
from sklearn.metrics import confusion_matrix, ConfusionMatrixDisplay
#confusion matrix
cm = confusion_matrix(y_test, y_pred_rf)
disp = ConfusionMatrixDisplay(confusion_matrix=cm, display_labels=["Not Top Seller", "Top Seller"])

#heatmap
disp.plot(cmap=plt.cm.Blues)
plt.title("Random Forest Confusion Matrix")
plt.show()

In [ ]:
from sklearn.model_selection import GridSearchCV
from sklearn.ensemble import RandomForestClassifier
rf = RandomForestClassifier(random_state=42, class_weight='balanced')
param_grid = {
    'n_estimators': [100, 200, 300],
    'max_depth': [5, 10, 20, None]
}

# grid search
grid_search = GridSearchCV(estimator=rf, param_grid=param_grid, 
                           cv=5, scoring='f1', n_jobs=-1, verbose=2)

# fit model
grid_search.fit(X_train, y_train)

# best model
best_rf = grid_search.best_estimator_
print("Best Parameters:", grid_search.best_params_)

# predict and evaluate again
y_pred_best_rf = best_rf.predict(X_test)

from sklearn.metrics import classification_report
print("Classification Report for Tuned Random Forest:")
print(classification_report(y_test, y_pred_best_rf))

In [ ]:
# distribution of global sales
plt.figure(figsize=(8, 5))
sns.histplot(df_clean['Global_Sales'], bins=10, kde=True)
plt.title('Distribution of Global Sales')
plt.xlabel('Global Sales (Millions)')
plt.ylabel('Frequency')
plt.tight_layout()
plt.show()

In [ ]:
# proportion of Top Sellers by genre
top_sellers_by_genre = df_clean[df_clean['Top_Seller_75'] == 1]['Genre'].value_counts(normalize=True) * 100
plt.figure(figsize=(8, 5))
top_sellers_by_genre.plot(kind='bar')
plt.title('Top Sellers by Genre (Proportion)')
plt.xlabel('Genre')
plt.ylabel('Percentage of Top Sellers')
plt.tight_layout()
plt.show()

In [ ]:
plt.figure(figsize=(8, 5))
sns.histplot(df_clean[df_clean['Global_Sales'] < 5]['Global_Sales'], bins=50, kde=True)
plt.title('Distribution of Global Sales (Under 5M Units)')
plt.xlabel('Global Sales (Millions)')
plt.ylabel('Frequency')
plt.tight_layout()
plt.show()


In [ ]:
# feature importance 
importances = best_rf.feature_importances_
features = X.columns
importance_df = pd.DataFrame({'Feature': features, 'Importance': importances})
importance_df = importance_df.sort_values('Importance', ascending=False)

plt.figure(figsize=(10, 6))
sns.barplot(x='Importance', y='Feature', data=importance_df, palette='crest')
plt.title("Feature Importance (Random Forest - Tuned)")
plt.tight_layout()
plt.show()

In [ ]:
# confusion matrix 
y_pred_best_rf = best_rf.predict(X_test)

ConfusionMatrixDisplay.from_predictions(y_test, y_pred_best_rf)
plt.title('Confusion Matrix (Tuned Random Forest)')
plt.tight_layout()
plt.show()